# ⚡ 02. Temporal, Cyclical & Autoregressive Feature Engineering

**Goal**: Construct 35+ engineered features without lookahead leakage (shift $\ge 24$h).

In [ ]:
import pandas as pd
import numpy as np

# Load cleaned series
df = pd.read_csv("PJME_hourly.csv", parse_dates=['Datetime'], index_col='Datetime').sort_index()

# 1. Calendar Features
idx = df.index
df['hour'] = idx.hour
df['dayofweek'] = idx.dayofweek
df['month'] = idx.month
df['quarter'] = idx.quarter
df['dayofyear'] = idx.dayofyear
df['weekofyear'] = idx.isocalendar().week.astype(int)
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
df['is_business_hour'] = ((df['hour'] >= 8) & (df['hour'] <= 18) & (df['is_weekend'] == 0)).astype(int)

# 2. Cyclical Fourier Encodings (smooth circular boundary)
df['hour_sin'] = np.sin(2 * np.pi * idx.hour / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * idx.hour / 24.0)
df['dow_sin'] = np.sin(2 * np.pi * idx.dayofweek / 7.0)
df['dow_cos'] = np.cos(2 * np.pi * idx.dayofweek / 7.0)
df['month_sin'] = np.sin(2 * np.pi * idx.month / 12.0)
df['month_cos'] = np.cos(2 * np.pi * idx.month / 12.0)
df['doy_sin'] = np.sin(2 * np.pi * idx.dayofyear / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * idx.dayofyear / 365.25)

# 3. Autoregressive Lags (Shift >= 24h to prevent 24h horizon leakage)
for lag in [24, 48, 168, 336, 8760]:
    df[f'lag_{lag}h'] = df['PJME_MW'].shift(lag)

# 4. Rolling Statistics (Shifted by 24h)
target_shifted = df['PJME_MW'].shift(24)
for w in [24, 48, 168]:
    df[f'rolling_mean_{w}h'] = target_shifted.rolling(w).mean()
    df[f'rolling_std_{w}h'] = target_shifted.rolling(w).std()
    df[f'rolling_max_{w}h'] = target_shifted.rolling(w).max()
    df[f'rolling_min_{w}h'] = target_shifted.rolling(w).min()

# 5. Exponential Weighted Moving Averages
for alpha in [0.3, 0.1, 0.05]:
    df[f'ewm_alpha_{str(alpha).replace(".", "_")}'] = target_shifted.ewm(alpha=alpha).mean()

# 6. Interaction Features
df['delta_24h'] = df['PJME_MW'].shift(24) - df['PJME_MW'].shift(48)
df['hour_x_dow'] = df['hour'] * df['dayofweek']
df['biz_x_lag24'] = df['is_business_hour'] * df['lag_24h']

print(f"Total features constructed: {df.shape[1] - 1}")
df.dropna().head()
